In [ ]:
import pandas as pd

# قراءة البيانات
hosts_df = pd.read_csv('/content/olympic_hosts.csv')
results_df = pd.read_csv('/content/olympic_results.csv')

# عرض معلومات عامة عن ملف المستضيفين
print("=== معلومات ملف المستضيفين (Hosts) ===")
print(hosts_df.info())
display(hosts_df.head())

# عرض معلومات عامة عن ملف النتائج
print("\n=== معلومات ملف النتائج (Results) ===")
print(results_df.info())
display(results_df.head())

=== معلومات ملف المستضيفين (Hosts) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   game_slug        53 non-null     object
 1   game_end_date    53 non-null     object
 2   game_start_date  53 non-null     object
 3   game_location    53 non-null     object
 4   game_name        53 non-null     object
 5   game_season      53 non-null     object
 6   game_year        53 non-null     int64 
dtypes: int64(1), object(6)
memory usage: 3.0+ KB
None


,game_slug,game_end_date,game_start_date,game_location,game_name,game_season,game_year
0,beijing-2022,2022-02-20T12:00:00Z,2022-02-04T15:00:00Z,China,Beijing 2022,Winter,2022
1,tokyo-2020,2021-08-08T14:00:00Z,2021-07-23T11:00:00Z,Japan,Tokyo 2020,Summer,2020
2,pyeongchang-2018,2018-02-25T08:00:00Z,2018-02-08T23:00:00Z,Republic of Korea,PyeongChang 2018,Winter,2018
3,rio-2016,2016-08-21T21:00:00Z,2016-08-05T12:00:00Z,Brazil,Rio 2016,Summer,2016
4,sochi-2014,2014-02-23T16:00:00Z,2014-02-07T04:00:00Z,Russian Federation,Sochi 2014,Winter,2014



=== معلومات ملف النتائج (Results) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162804 entries, 0 to 162803
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   discipline_title       162804 non-null  object
 1   event_title            162804 non-null  object
 2   slug_game              162804 non-null  object
 3   participant_type       162804 non-null  object
 4   medal_type             20206 non-null   object
 5   athletes               7976 non-null    object
 6   rank_equal             32526 non-null   object
 7   rank_position          158926 non-null  object
 8   country_name           162804 non-null  object
 9   country_code           157768 non-null  object
 10  country_3_letter_code  162804 non-null  object
 11  athlete_url            129991 non-null  object
 12  athlete_full_name      141646 non-null  object
 13  value_unit             78646 non-null   object
 14  value_type   

,discipline_title,event_title,slug_game,participant_type,medal_type,athletes,rank_equal,rank_position,country_name,country_code,country_3_letter_code,athlete_url,athlete_full_name,value_unit,value_type
0,Curling,Mixed Doubles,beijing-2022,GameTeam,GOLD,"[('Stefania CONSTANTINI', 'https://olympics.co...",False,1,Italy,IT,ITA,NaN,NaN,NaN,NaN
1,Curling,Mixed Doubles,beijing-2022,GameTeam,SILVER,"[('Kristin SKASLIEN', 'https://olympics.com/en...",False,2,Norway,NO,NOR,NaN,NaN,NaN,NaN
2,Curling,Mixed Doubles,beijing-2022,GameTeam,BRONZE,"[('Almida DE VAL', 'https://olympics.com/en/at...",False,3,Sweden,SE,SWE,NaN,NaN,NaN,NaN
3,Curling,Mixed Doubles,beijing-2022,GameTeam,NaN,"[('Jennifer DODDS', 'https://olympics.com/en/a...",False,4,Great Britain,GB,GBR,NaN,NaN,NaN,NaN
4,Curling,Mixed Doubles,beijing-2022,GameTeam,NaN,"[('Rachel HOMAN', 'https://olympics.com/en/ath...",False,5,Canada,CA,CAN,NaN,NaN,NaN,NaN


In [ ]:
# التحقق من القيم المفقودة (Null values)
print("=== القيم المفقودة في ملف المستضيفين ===")
print(hosts_df.isnull().sum())

print("\n=== القيم المفقودة في ملف النتائج ===")
print(results_df.isnull().sum())

=== القيم المفقودة في ملف المستضيفين ===
game_slug          0
game_end_date      0
game_start_date    0
game_location      0
game_name          0
game_season        0
game_year          0
dtype: int64

=== القيم المفقودة في ملف النتائج ===
discipline_title              0
event_title                   0
slug_game                     0
participant_type              0
medal_type               142598
athletes                 154828
rank_equal               130278
rank_position              3878
country_name                  0
country_code               5036
country_3_letter_code         0
athlete_url               32813
athlete_full_name         21158
value_unit                84158
value_type                72755
dtype: int64


In [ ]:
# 1. تصفية المستضيفين للألعاب الشتوية فقط
winter_hosts = hosts_df[hosts_df['game_season'] == 'Winter'][['game_slug', 'game_year', 'game_location']]

# 2. دمج النتائج مع المستضيفين للحصول على السنة والموسم
winter_results = results_df.merge(winter_hosts, left_on='slug_game', right_on='game_slug', how='inner')

# 3. ملء القيم الفارغة في الميداليات بـ 'No Medal'
winter_results['medal_type'] = winter_results['medal_type'].fillna('No Medal')

# 4. حساب عدد الميداليات لكل دولة في كل دورة ألعاب شتوية
# سنقوم بحساب الميداليات الذهبية والفضية والبرونزية بشكل منفصل ثم جمعها
medal_counts = winter_results.groupby(['game_year', 'country_3_letter_code', 'country_name', 'medal_type']).size().unstack(fill_value=0).reset_index()

# التأكد من وجود أعمدة الميداليات الثلاثة حتى لو لم تحصل بعض الدول عليها
for medal in ['GOLD', 'SILVER', 'BRONZE']:
    if medal not in medal_counts.columns:
        medal_counts[medal] = 0

# حساب إجمالي الميداليات
medal_counts['total_medals'] = medal_counts['GOLD'] + medal_counts['SILVER'] + medal_counts['BRONZE']

# ترتيب البيانات حسب السنة وإجمالي الميداليات تنازلياً
medal_counts = medal_counts.sort_values(by=['game_year', 'total_medals'], ascending=[True, False]).reset_index(drop=True)

print("=== عينة من البيانات بعد التنظيف والتحضير ===")
display(medal_counts.head(15))
print(f"\nأبعاد البيانات الجاهزة للنمذجة: {medal_counts.shape}")

=== عينة من البيانات بعد التنظيف والتحضير ===


medal_type,game_year,country_3_letter_code,country_name,BRONZE,GOLD,No Medal,SILVER,total_medals
0,1924,NOR,Norway,6,4,19,7,17
1,1924,FIN,Finland,3,4,22,4,11
2,1924,GBR,Great Britain,2,1,35,1,4
3,1924,USA,United States of America,1,1,33,2,4
4,1924,AUT,Austria,0,2,0,1,3
5,1924,FRA,France,3,0,42,0,3
6,1924,SUI,Switzerland,1,2,23,0,3
7,1924,SWE,Sweden,0,1,29,1,2
8,1924,BEL,Belgium,1,0,22,0,1
9,1924,CAN,Canada,0,1,7,0,1



أبعاد البيانات الجاهزة للنمذجة: (1211, 8)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. إعداد قائمة بالدول المستضيفة لكل دورة شتوية لربطها بالبيانات
host_countries_map = {
    1992: 'FRA', # Albertville
    1994: 'NOR', # Lillehammer
    1998: 'JPN', # Nagano
    2002: 'USA', # Salt Lake City
    2006: 'ITA', # Turin
    2010: 'CAN', # Vancouver
    2014: 'RUS', # Sochi
    2018: 'KOR', # PyeongChang
    2022: 'CHN', # Beijing
    2026: 'ITA'  # Milano-Cortina
}

# 2. فلترة البيانات من عام 1992 فصاعداً
winter_data = medal_counts[medal_counts['game_year'] >= 1992].copy()

# إضافة عمود يوضح إذا كانت الدولة هي المستضيفة في تلك السنة
winter_data['is_host'] = winter_data.apply(
    lambda row: 1 if host_countries_map.get(row['game_year']) == row['country_3_letter_code'] else 0,
    axis=1
)

# 3. إنشاء ميزة الميداليات السابقة (Lagged Medals)
# سنرتب البيانات حسب الدولة والسنة لضمان التسلسل الصحيح
winter_data = winter_data.sort_values(by=['country_3_letter_code', 'game_year']).reset_index(drop=True)
winter_data['prev_total_medals'] = winter_data.groupby('country_3_letter_code')['total_medals'].shift(1)

# ملء الميداليات السابقة للدول المشاركة لأول مرة بـ 0
winter_data['prev_total_medals'] = winter_data['prev_total_medals'].fillna(0)

# عرض عينة للتحقق من الميزات الجديدة
display(winter_data[['game_year', 'country_name', 'total_medals', 'prev_total_medals', 'is_host']].head(10))

medal_type,game_year,country_name,total_medals,prev_total_medals,is_host
0,1992,Netherlands Antilles,0,0.0,0
1,2006,Albania,0,0.0,0
2,2010,Albania,0,0.0,0
3,2014,Albania,0,0.0,0
4,2018,Albania,0,0.0,0
5,2022,Albania,0,0.0,0
6,1992,Algeria,0,0.0,0
7,2006,Algeria,0,0.0,0
8,2010,Algeria,0,0.0,0
9,1992,Andorra,0,0.0,0


In [ ]:
# 4. فصل البيانات إلى تدريب واختبار (استخدام عام 2022 كـ Holdout Test Set)
train_set = winter_data[winter_data['game_year'] < 2022]
test_set = winter_data[winter_data['game_year'] == 2022]

# ميزات النموذج والهدف المستهدف
features = ['prev_total_medals', 'is_host']
target = 'total_medals'

X_train = train_set[features]
y_train = train_set[target]
X_test = test_set[features]
y_test = test_set[target]

# تدريب النموذج (Random Forest Regressor)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# التنبؤ على مجموعة الاختبار وتقييم الأداء
predictions_2022 = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions_2022)
r2 = r2_score(y_test, predictions_2022)

print("=== تقييم النموذج على دورة 2022 (Holdout Test Set) ===")
print(f"متوسط الخطأ المطلق (MAE): {mae:.2f} ميدالية")
print(f"معامل التحديد (R² Score): {r2:.2f}")

=== تقييم النموذج على دورة 2022 (Holdout Test Set) ===
متوسط الخطأ المطلق (MAE): 1.72 ميدالية
معامل التحديد (R² Score): 0.73


In [ ]:
# 5. تدريب النموذج النهائي على كامل البيانات المتاحة لرفع دقة توقعات 2026
X_all = winter_data[features]
y_all = winter_data[target]
final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_all, y_all)

# 6. تجهيز بيانات التنبؤ لدورة 2026
# سنأخذ جميع الدول الفريدة التي شاركت في دورة 2022
countries_2022 = winter_data[winter_data['game_year'] == 2022].copy()

# بناء جدول التوقعات لعام 2026
predict_2026 = pd.DataFrame({
    'country_3_letter_code': countries_2022['country_3_letter_code'],
    'country_name': countries_2022['country_name'],
    'prev_total_medals': countries_2022['total_medals']  # الميداليات السابقة لـ 2026 هي نتائج 2022
})

# تحديد إيطاليا كمستضيفة لعام 2026 وتصفير المستضيف للآخرين
predict_2026['is_host'] = predict_2026['country_3_letter_code'].apply(lambda x: 1 if x == 'ITA' else 0)

# 7. التنبؤ الأولي لعام 2026
raw_preds = final_model.predict(predict_2026[features])
predict_2026['raw_prediction'] = np.clip(raw_preds, 0, None)  # منع القيم السالبة

# 8. تعديل وضبط المجموع ليكون بالضبط 348 ميدالية (3 * 116)
target_sum = 116 * 3
current_sum = predict_2026['raw_prediction'].sum()

# التعديل بالنسب والتوزيع المتساوي لكسر الفروق البسيطة
predict_2026['final_prediction'] = (predict_2026['raw_prediction'] / current_sum) * target_sum
predict_2026['final_prediction'] = predict_2026['final_prediction'].round().astype(int)

# تعديل الفروقات البسيطة الناتجة عن التقريب لضمان أن المجموع الكلي يساوي 348 بدقة
difference = target_sum - predict_2026['final_prediction'].sum()
if difference != 0:
    # نقوم بتوزيع الفرق على أعلى الدول توقعاً للمحافظة على التناسب المنطقي
    top_indices = predict_2026.sort_values(by='final_prediction', ascending=False).index
    for i in range(abs(difference)):
        idx = top_indices[i % len(top_indices)]
        predict_2026.loc[idx, 'final_prediction'] += 1 if difference > 0 else -1

# عرض النتائج النهائية للتنبؤات لعام 2026 مرتبة تنازلياً حسب التنبؤ الأخير
print(f"\n=== التوقعات النهائية لدورة الألعاب الشتوية 2026 (إجمالي الميداليات: {predict_2026['final_prediction'].sum()}) ===")
display(predict_2026.sort_values(by='final_prediction', ascending=False).head(20))


=== التوقعات النهائية لدورة الألعاب الشتوية 2026 (إجمالي الميداليات: 348) ===


,country_3_letter_code,country_name,prev_total_medals,is_host,raw_prediction,final_prediction
515,NOR,Norway,37,0,28.580000,27
259,GER,Germany,27,0,28.090370,26
124,CAN,Canada,26,0,28.242370,26
700,USA,United States of America,25,0,27.582833,26
344,ITA,Italy,17,1,26.612119,25
566,ROC,ROC,32,0,25.093000,23
500,NED,Netherlands,17,0,17.116167,16
362,JPN,Japan,18,0,15.012655,13
233,FRA,France,14,0,14.007114,13
144,CHN,People's Republic of China,15,0,14.790042,13


### ربط Google Drive وحفظ النموذج التنبئي لعام 2026

In [ ]:
from google.colab import drive
import joblib
import os

# 1. ربط Google Drive
drive.mount('/content/drive')

# 2. تحديد مسار المجلد وحفظ النموذج
drive_folder = '/content/drive/MyDrive/Olympic_Predictions'
os.makedirs(drive_folder, exist_ok=True)  # إنشاء المجلد إذا لم يكن موجوداً

model_path = os.path.join(drive_folder, 'winter_olympics_2026_model.pkl')
joblib.dump(final_model, model_path)

print(f"تم حفظ النموذج بنجاح في المسار التالي:\n{model_path}")

MessageError: Error: credential propagation was unsuccessful

### حل بديل: حفظ النموذج محلياً في Colab لتحميله لجهازك

In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

try:
    # 1. قراءة وتنظيف البيانات من الصفر لضمان عمل الخلية بشكل مستقل تماماً
    print("إعادة تحميل البيانات وتجهيزها...")
    hosts_df = pd.read_csv('/content/olympic_hosts.csv')
    results_df = pd.read_csv('/content/olympic_results.csv')

    # تصفية وتحضير البيانات
    winter_hosts = hosts_df[hosts_df['game_season'] == 'Winter'][['game_slug', 'game_year', 'game_location']]
    winter_results = results_df.merge(winter_hosts, left_on='slug_game', right_on='game_slug', how='inner')
    winter_results['medal_type'] = winter_results['medal_type'].fillna('No Medal')

    # تجميع الميداليات
    medal_counts = winter_results.groupby(['game_year', 'country_3_letter_code', 'country_name', 'medal_type']).size().unstack(fill_value=0).reset_index()
    for medal in ['GOLD', 'SILVER', 'BRONZE']:
        if medal not in medal_counts.columns:
            medal_counts[medal] = 0
    medal_counts['total_medals'] = medal_counts['GOLD'] + medal_counts['SILVER'] + medal_counts['BRONZE']

    # 2. بناء الميزات وتدريب النموذج
    print("إعادة تدريب النموذج...")
    host_countries_map = {1992: 'FRA', 1994: 'NOR', 1998: 'JPN', 2002: 'USA', 2006: 'ITA', 2010: 'CAN', 2014: 'RUS', 2018: 'KOR', 2022: 'CHN', 2026: 'ITA'}
    winter_data = medal_counts[medal_counts['game_year'] >= 1992].copy()
    winter_data['is_host'] = winter_data.apply(lambda row: 1 if host_countries_map.get(row['game_year']) == row['country_3_letter_code'] else 0, axis=1)
    winter_data = winter_data.sort_values(by=['country_3_letter_code', 'game_year']).reset_index(drop=True)
    winter_data['prev_total_medals'] = winter_data.groupby('country_3_letter_code')['total_medals'].shift(1).fillna(0)

    features = ['prev_total_medals', 'is_host']
    target = 'total_medals'
    X_all = winter_data[features]
    y_all = winter_data[target]

    final_model = RandomForestRegressor(n_estimators=100, random_state=42)
    final_model.fit(X_all, y_all)

    # 3. حفظ النموذج محلياً
    local_path = '/content/winter_olympics_2026_model.pkl'
    joblib.dump(final_model, local_path)
    print(f"\nتم حفظ النموذج محلياً بنجاح في: {local_path}")
    print("يمكنك الآن تحميله مباشرة بالضغط عليه بزر الفأرة الأيمن في قائمة الملفات الجانبية واختيار Download.")

except Exception as e:
    print(f"حدث خطأ أثناء محاولة تدريب أو حفظ النموذج: {e}")

إعادة تحميل البيانات وتجهيزها...
إعادة تدريب النموذج...

تم حفظ النموذج محلياً بنجاح في: /content/winter_olympics_2026_model.pkl
يمكنك الآن تحميله مباشرة بالضغط عليه بزر الفأرة الأيمن في قائمة الملفات الجانبية واختيار Download.


In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import lightgbm as lgb

# 1. تجهيز ميزات متطورة (Feature Engineering) مع الحفاظ على القيم الكبرى للدول القوية
winter_data_sorted = winter_data.sort_values(by=['country_3_letter_code', 'game_year']).reset_index(drop=True)

# حساب المتوسط المتحرك للميداليات لآخر 3 دورات (3-games rolling average)
# نقوم باستخدام shift(1) لتفادي تسريب بيانات الدورة الحالية للتنبؤ
winter_data_sorted['rolling_avg_3_medals'] = winter_data_sorted.groupby('country_3_letter_code')['total_medals'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
).fillna(0)

# الميزات الجديدة المستهدفة للدراسة
advanced_features = ['prev_total_medals', 'is_host', 'rolling_avg_3_medals']
target = 'total_medals'

# تقسيم البيانات إلى تدريب واختبار (2022 كـ Holdout)
train_df = winter_data_sorted[winter_data_sorted['game_year'] < 2022]
test_df = winter_data_sorted[winter_data_sorted['game_year'] == 2022]

X_train = train_df[advanced_features]
y_train = train_df[target]
X_test = test_df[advanced_features]
y_test = test_df[target]

print("=== مقارنة النماذج المتقدمة الأساسية ===")
# 2. مقارنة النماذج الأساسية قبل الضبط
xgb_model = xgb.XGBRegressor(random_state=42)
lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)

xgb_model.fit(X_train, y_train)
lgb_model.fit(X_train, y_train)

xgb_mae = mean_absolute_error(y_test, xgb_model.predict(X_test))
lgb_mae = mean_absolute_error(y_test, lgb_model.predict(X_test))

print(f"XGBoost Base MAE: {xgb_mae:.3f}")
print(f"LightGBM Base MAE: {lgb_mae:.3f}")

# 3. ضبط المعلمات الفائقة للنموذج الأفضل (سنقوم بضبط XGBoost لقدرته العالية على إدارة القيم الفائقة)
print("\n=== ضبط المعلمات الفائقة لـ XGBoost ===")
param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_xgb = random_search.best_estimator_
tuned_pred = best_xgb.predict(X_test)
tuned_mae = mean_absolute_error(y_test, tuned_pred)

# 4. مقارنة الأداء بين النموذج القديم والمطور
old_mae = 1.72
print("\n=== التقييم والمقارنة النهائية ===")
print(f"الخطأ القديم للنموذج (Old MAE): {old_mae:.2f} ميدالية")
print(f"الخطأ الجديد المطور (New MAE): {tuned_mae:.2f} ميدالية")
print(f"نسبة التحسن: {((old_mae - tuned_mae) / old_mae) * 100:.2f}%")

# حفظ النموذج الأفضل والمطور محلياً لاستخدامه لاحقاً
joblib.dump(best_xgb, '/content/winter_olympics_2026_model.pkl')
print("\nتم حفظ النموذج المحسّن الجديد بنجاح في: /content/winter_olympics_2026_model.pkl")

=== مقارنة النماذج المتقدمة الأساسية ===
XGBoost Base MAE: 1.858
LightGBM Base MAE: 1.616

=== ضبط المعلمات الفائقة لـ XGBoost ===

=== التقييم والمقارنة النهائية ===
الخطأ القديم للنموذج (Old MAE): 1.72 ميدالية
الخطأ الجديد المطور (New MAE): 1.63 ميدالية
نسبة التحسن: 5.09%

تم حفظ النموذج المحسّن الجديد بنجاح في: /content/winter_olympics_2026_model.pkl


In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb

# 1. تعريف فضاء البحث عن المعلمات الفائقة لـ LightGBM
param_dist_lgb = {
    'n_estimators': [50, 100, 150, 200, 300],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.15],
    'max_depth': [3, 4, 5, 6, 7, -1],
    'num_leaves': [7, 15, 31, 63],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [5, 10, 20, 30]
}

# 2. إعداد RandomizedSearchCV للبحث عن أقل MAE
lgb_search = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(random_state=42, verbose=-1),
    param_distributions=param_dist_lgb,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

# تدريب البحث على بيانات التدريب
lgb_search.fit(X_train, y_train)

# استخراج النموذج الأفضل
best_lgb = lgb_search.best_estimator_

# 3. تقييم النموذج المضبط على مجموعة الاختبار 2022
tuned_lgb_pred = best_lgb.predict(X_test)
tuned_lgb_mae = mean_absolute_error(y_test, tuned_lgb_pred)

print("=== نتائج ضبط نموذج LightGBM ===")
print(f"أفضل المعلمات الفائقة: {lgb_search.best_params_}")
print(f"الخطأ القديم للنموذج الأساسي (LGBM Baseline MAE): 1.616")
print(f"الخطأ الجديد بعد الضبط (Tuned LGBM MAE): {tuned_lgb_mae:.3f} ميدالية")

# 4. حفظ النموذج المضبط محلياً وفي Google Drive
local_model_path = '/content/winter_olympics_2026_model_v3.pkl'
joblib.dump(best_lgb, local_model_path)
print(f"\nتم حفظ النموذج محلياً في: {local_model_path}")

drive_model_path = '/content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_model_v3.pkl'
try:
    # إنشاء المجلد في Drive في حال لم يكن موجوداً ومتاحاً للوصول
    os.makedirs(os.path.dirname(drive_model_path), exist_ok=True)
    joblib.dump(best_lgb, drive_model_path)
    print(f"تم حفظ نسخة من النموذج في Google Drive بنجاح: {drive_model_path}")
except Exception as e:
    print(f"تنبيه: تعذر الحفظ في Google Drive (قد لا يكون متصلاً حالياً). التفاصيل: {e}")

=== نتائج ضبط نموذج LightGBM ===
أفضل المعلمات الفائقة: {'subsample': 0.6, 'num_leaves': 15, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
الخطأ القديم للنموذج الأساسي (LGBM Baseline MAE): 1.616
الخطأ الجديد بعد الضبط (Tuned LGBM MAE): 1.596 ميدالية

تم حفظ النموذج محلياً في: /content/winter_olympics_2026_model_v3.pkl
تم حفظ نسخة من النموذج في Google Drive بنجاح: /content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_model_v3.pkl


In [ ]:
import requests
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

# 1. جلب البيانات الاقتصادية (الناتج المحلي الإجمالي وعدد السكان) باستخدام API البنك الدولي
def get_world_bank_data(indicator, col_name, year):
    url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator}?date={year}&format=json&per_page=300"
    try:
        res = requests.get(url, timeout=10).json()
        if len(res) > 1:
            records = []
            for item in res[1]:
                country_code = item['countryiso3code']
                val = item['value']
                if country_code and val is not None:
                    records.append({'country_code': country_code, col_name: val})
            return pd.DataFrame(records)
    except Exception:
        pass
    return pd.DataFrame(columns=['country_code', col_name])

# جلب بيانات الناتج المحلي الإجمالي (GDP) وعدد السكان (Population) لعام 2021/2022 كمؤشر للقدرة الاقتصادية والرياضية
print("جاري جلب المؤشرات الاقتصادية من البنك الدولي...")
gdp_df = get_world_bank_data('NY.GDP.MKTP.CD', 'gdp', 2021)
pop_df = get_world_bank_data('SP.POP.TOTL', 'population', 2021)

# دمج البيانات الاقتصادية معاً
if not gdp_df.empty and not pop_df.empty:
    macro_df = pd.merge(gdp_df, pop_df, on='country_code', how='inner')
    macro_df['gdp_per_capita'] = macro_df['gdp'] / macro_df['population']
else:
    print("تعذر جلب البيانات التلقائية، جاري توليد قيم محاكاة عالية الدقة مبنية على معايير البنك الدولي...")
    macro_df = pd.DataFrame(columns=['country_code', 'gdp', 'population', 'gdp_per_capita'])

# معايير افتراضية عالية الدقة للدول الكبرى في الألعاب الشتوية في حال الفشل أو كاحتياط
default_macro = {
    'USA': (23.3e12, 332e6),
    'CHN': (17.7e12, 1412e6),
    'GER': (4.26e12, 83.2e6),
    'NOR': (482e9, 5.4e6),
    'CAN': (1.98e12, 38.2e6),
    'RUS': (1.78e12, 143e6),
    'ROC': (1.78e12, 143e6), # روسيا تحت مسمى اللجنة الأولمبية
    'FRA': (2.95e12, 67.7e6),
    'ITA': (2.1e12, 59.1e6),
    'JPN': (4.93e12, 125.7e6),
    'KOR': (1.81e12, 51.7e6),
    'AUT': (480e9, 8.9e6),
    'SWE': (635e9, 10.4e6),
    'SUI': (812e9, 8.7e6),
    'NED': (1.01e12, 17.5e6),
    'FIN': (297e9, 5.5e6)
}

# 2. دمج الميزات الاقتصادية مع البيانات الحالية
winter_data_macro = winter_data_sorted.copy()

def apply_macro_features(row):
    cc = row['country_3_letter_code']
    gdp, pop = np.nan, np.nan

    # البحث في بيانات البنك الدولي أولاً
    if not macro_df.empty and cc in macro_df['country_code'].values:
        match = macro_df[macro_df['country_code'] == cc].iloc[0]
        gdp, pop = match['gdp'], match['population']

    # استخدام الاحتياط إذا كانت البيانات مفقودة
    if (pd.isna(gdp) or pd.isna(pop)) and cc in default_macro:
        gdp, pop = default_macro[cc]

    # في حال لم تكن دولة كبرى نضع قيماً متوسطة افتراضية
    if pd.isna(gdp):
        gdp = 50e9  # 50 مليار
        pop = 10e6  # 10 مليون

    return pd.Series([gdp, pop, gdp/pop])

winter_data_macro[['gdp', 'population', 'gdp_per_capita']] = winter_data_macro.apply(apply_macro_features, axis=1)

# تطبيق مقياس لوغاريتمي على الأرقام الضخمة لتحسين أداء النموذج الشجري
winter_data_macro['log_gdp'] = np.log1p(winter_data_macro['gdp'])
winter_data_macro['log_population'] = np.log1p(winter_data_macro['population'])
winter_data_macro['log_gdp_per_capita'] = np.log1p(winter_data_macro['gdp_per_capita'])

# تحديث قائمة الميزات لتشمل الميزات الاقتصادية الذكية
ultimate_features = advanced_features + ['log_gdp', 'log_population', 'log_gdp_per_capita']

# تقسيم البيانات
train_df_m = winter_data_macro[winter_data_macro['game_year'] < 2022]
test_df_m = winter_data_macro[winter_data_macro['game_year'] == 2022]

X_train_m = train_df_m[ultimate_features]
y_train_m = train_df_m[target]
X_test_m = test_df_m[ultimate_features]
y_test_m = test_df_m[target]

# 3. إعادة تدريب نموذج LightGBM المضبط على الميزات الجديدة
best_params = {
    'subsample': 0.6, 'num_leaves': 15, 'n_estimators': 200,
    'min_child_samples': 20, 'max_depth': 6, 'learning_rate': 0.05,
    'colsample_bytree': 0.6, 'random_state': 42, 'verbose': -1
}

ultimate_lgb = lgb.LGBMRegressor(**best_params)
ultimate_lgb.fit(X_train_m, y_train_m)

# 4. التقييم على مجموعة الاختبار 2022 ومقارنة النتائج
ultimate_preds = ultimate_lgb.predict(X_test_m)
ultimate_mae = mean_absolute_error(y_test_m, ultimate_preds)

previous_best = 1.596
print("\n=== نتائج إدخال المؤشرات الاقتصادية (GDP & Population) ===")
print(f"الخطأ السابق بدون الميزات الاقتصادية: {previous_best:.3f} ميدالية")
print(f"الخطأ الجديد مع الميزات الاقتصادية: {ultimate_mae:.3f} ميدالية")

# 5. حفظ النموذج الأفضل إذا تم تحسين دقة التوقع
if ultimate_mae < previous_best:
    print("\n🔥 رائع! الميزات الاقتصادية رفعت من دقة النموذج وخفضت الخطأ المالي المطلق.")
    joblib.dump(ultimate_lgb, '/content/winter_olympics_2026_model_ultimate.pkl')
    print("تم حفظ النموذج النهائي بنجاح باسم: 'winter_olympics_2026_model_ultimate.pkl'")
else:
    print("\nالبيانات الاقتصادية لم توفر تغيراً هائلاً على هذا التقسيم، لكن قمنا بحفظ النموذج الاحتياطي المطور كـ 'winter_olympics_2026_model_ultimate.pkl'.")
    joblib.dump(ultimate_lgb, '/content/winter_olympics_2026_model_ultimate.pkl')


جاري جلب المؤشرات الاقتصادية من البنك الدولي...

=== نتائج إدخال المؤشرات الاقتصادية (GDP & Population) ===
الخطأ السابق بدون الميزات الاقتصادية: 1.596 ميدالية
الخطأ الجديد مع الميزات الاقتصادية: 1.334 ميدالية

🔥 رائع! الميزات الاقتصادية رفعت من دقة النموذج وخفضت الخطأ المالي المطلق.
تم حفظ النموذج النهائي بنجاح باسم: 'winter_olympics_2026_model_ultimate.pkl'


In [ ]:
import numpy as np
import pandas as pd
import joblib
import os

# 1. تجهيز ميزات توقعات عام 2026 بناءً على نتائج عام 2022 والميزات المتقدمة والمؤشرات الاقتصادية
countries_2022 = winter_data_macro[winter_data_macro['game_year'] == 2022].copy()

predict_2026 = pd.DataFrame({
    'country_3_letter_code': countries_2022['country_3_letter_code'],
    'country_name': countries_2022['country_name'],
    'prev_total_medals': countries_2022['total_medals'],
    'is_host': countries_2022['country_3_letter_code'].apply(lambda x: 1 if x == 'ITA' else 0),
    # المتوسط المتحرك لآخر 3 دورات (2014، 2018، 2022) لحساب ميزات 2026
    'rolling_avg_3_medals': countries_2022.apply(
        lambda row: winter_data_macro[
            (winter_data_macro['country_3_letter_code'] == row['country_3_letter_code']) &
            (winter_data_macro['game_year'] <= 2022)
        ]['total_medals'].tail(3).mean(),
        axis=1
    ).fillna(0),
    'log_gdp': countries_2022['log_gdp'],
    'log_population': countries_2022['log_population'],
    'log_gdp_per_capita': countries_2022['log_gdp_per_capita']
})

# 2. تطبيق التنبؤ باستخدام النموذج النهائي Ultimate LightGBM
raw_preds_2026 = ultimate_lgb.predict(predict_2026[ultimate_features])
predict_2026['raw_prediction'] = np.clip(raw_preds_2026, 0, None)  # منع القيم السالبة

# 3. ضبط المجموع الكلي للميداليات ليكون 348 بدقة (116 حدث رياضي * 3)
target_sum = 116 * 3
current_sum = predict_2026['raw_prediction'].sum()
predict_2026['final_prediction'] = (predict_2026['raw_prediction'] / current_sum) * target_sum
predict_2026['final_prediction'] = predict_2026['final_prediction'].round().astype(int)

# كسر الفروقات الطفيفة في التقريب لضمان أن المجموع = 348
difference = target_sum - predict_2026['final_prediction'].sum()
if difference != 0:
    top_indices = predict_2026.sort_values(by='final_prediction', ascending=False).index
    for i in range(abs(difference)):
        idx = top_indices[i % len(top_indices)]
        predict_2026.loc[idx, 'final_prediction'] += 1 if difference > 0 else -1

# ترتيب وعرض أفضل 20 دولة
leaderboard_2026 = predict_2026.sort_values(by='final_prediction', ascending=False).copy()
leaderboard_display = leaderboard_2026.head(20)[[
    'country_name', 'country_3_letter_code', 'prev_total_medals', 'final_prediction'
]]
leaderboard_display.columns = ['الدولة', 'رمز الدولة', 'ميداليات 2022', 'التوقع النهائي لـ 2026']

print("=== جدول التوقعات النهائي لدورة الألعاب الأولمبية الشتوية 2026 ===")
display(leaderboard_display)

# 4. حفظ جدول التوقعات النهائي كملف CSV محلياً وفي Google Drive
local_csv_path = '/content/winter_olympics_2026_predictions.csv'
leaderboard_2026.to_csv(local_csv_path, index=False)
print(f"\nتم حفظ التوقعات محلياً في: {local_csv_path}")

drive_csv_path = '/content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_predictions.csv'
try:
    os.makedirs(os.path.dirname(drive_csv_path), exist_ok=True)
    leaderboard_2026.to_csv(drive_csv_path, index=False)
    print(f"تم حفظ التوقعات في Google Drive بنجاح: {drive_csv_path}")
except Exception as e:
    print(f"تنبيه: تعذر حفظ الملف في Google Drive. التفاصيل: {e}")

=== جدول التوقعات النهائي لدورة الألعاب الأولمبية الشتوية 2026 ===


,الدولة,رمز الدولة,ميداليات 2022,التوقع النهائي لـ 2026
259,Germany,GER,27,31
700,United States of America,USA,25,31
515,Norway,NOR,37,29
124,Canada,CAN,26,28
566,ROC,ROC,32,24
500,Netherlands,NED,17,22
233,France,FRA,14,20
625,Switzerland,SUI,14,20
362,Japan,JPN,18,20
642,Sweden,SWE,18,18



تم حفظ التوقعات محلياً في: /content/winter_olympics_2026_predictions.csv
تم حفظ التوقعات في Google Drive بنجاح: /content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_predictions.csv


### كود تحميل واستخدام النموذج لاحقاً

In [ ]:
# لاستعادة النموذج واستخدامه في المستقبل، يمكنك تشغيل الكود التالي:
"""
import joblib
from google.colab import drive

# ربط الـ Drive أولاً
drive.mount('/content/drive')

# تحميل النموذج
loaded_model = joblib.load('/content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_model.pkl')
print("تم تحميل النموذج بنجاح وهو جاهز للاستخدام!")
"""

'\nimport joblib\nfrom google.colab import drive\n\n# ربط الـ Drive أولاً\ndrive.mount(\'/content/drive\')\n\n# تحميل النموذج\nloaded_model = joblib.load(\'/content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_model.pkl\')\nprint("تم تحميل النموذج بنجاح وهو جاهز للاستخدام!")\n'

In [ ]:
# قراءة الملفات المرفوعة، التحقق من النموذج أو تدريبه وحفظه، ثم تطبيق التنبؤ لعام 2026
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.ensemble import RandomForestRegressor

# 1. قراءة وتحضير البيانات من الملفات المرفوعة
hosts_df = pd.read_csv('/content/olympic_hosts.csv')
results_df = pd.read_csv('/content/olympic_results.csv')

winter_hosts = hosts_df[hosts_df['game_season'] == 'Winter'][['game_slug', 'game_year', 'game_location']]
winter_results = results_df.merge(winter_hosts, left_on='slug_game', right_on='game_slug', how='inner')
winter_results['medal_type'] = winter_results['medal_type'].fillna('No Medal')

medal_counts = winter_results.groupby(['game_year', 'country_3_letter_code', 'country_name', 'medal_type']).size().unstack(fill_value=0).reset_index()
for medal in ['GOLD', 'SILVER', 'BRONZE']:
    if medal not in medal_counts.columns:
        medal_counts[medal] = 0
medal_counts['total_medals'] = medal_counts['GOLD'] + medal_counts['SILVER'] + medal_counts['BRONZE']

host_countries_map = {1992: 'FRA', 1994: 'NOR', 1998: 'JPN', 2002: 'USA', 2006: 'ITA', 2010: 'CAN', 2014: 'RUS', 2018: 'KOR', 2022: 'CHN', 2026: 'ITA'}
winter_data = medal_counts[medal_counts['game_year'] >= 1992].copy()
winter_data['is_host'] = winter_data.apply(lambda row: 1 if host_countries_map.get(row['game_year']) == row['country_3_letter_code'] else 0, axis=1)
winter_data = winter_data.sort_values(by=['country_3_letter_code', 'game_year']).reset_index(drop=True)
winter_data['prev_total_medals'] = winter_data.groupby('country_3_letter_code')['total_medals'].shift(1).fillna(0)

# 2. التحقق من وجود ملف النموذج، وفي حال عدم وجوده يتم تدريبه وحفظه تلقائياً
model_path = '/content/winter_olympics_2026_model.pkl'
features = ['prev_total_medals', 'is_host']
target = 'total_medals'

if os.path.exists(model_path):
    loaded_model = joblib.load(model_path)
    print("تم تحميل النموذج المحفوظ بنجاح!")
else:
    print("لم يتم العثور على ملف النموذج المحفوظ. جاري تدريب نموذج جديد وحفظه...")
    X_all = winter_data[features]
    y_all = winter_data[target]
    loaded_model = RandomForestRegressor(n_estimators=100, random_state=42)
    loaded_model.fit(X_all, y_all)
    joblib.dump(loaded_model, model_path)
    print(f"تم تدريب وحفظ النموذج الجديد بنجاح في: {model_path}")

# 3. تجهيز بيانات التنبؤ لعام 2026
countries_2022 = winter_data[winter_data['game_year'] == 2022].copy()
predict_2026 = pd.DataFrame({
    'country_3_letter_code': countries_2022['country_3_letter_code'],
    'country_name': countries_2022['country_name'],
    'prev_total_medals': countries_2022['total_medals']
})
predict_2026['is_host'] = predict_2026['country_3_letter_code'].apply(lambda x: 1 if x == 'ITA' else 0)

# 4. تطبيق التنبؤ
raw_preds = loaded_model.predict(predict_2026[features])
predict_2026['raw_prediction'] = np.clip(raw_preds, 0, None)

# ضبط المجموع ليكون 348 ميدالية بدقة
target_sum = 116 * 3
current_sum = predict_2026['raw_prediction'].sum()
predict_2026['final_prediction'] = (predict_2026['raw_prediction'] / current_sum) * target_sum
predict_2026['final_prediction'] = predict_2026['final_prediction'].round().astype(int)

difference = target_sum - predict_2026['final_prediction'].sum()
if difference != 0:
    top_indices = predict_2026.sort_values(by='final_prediction', ascending=False).index
    for i in range(abs(difference)):
        idx = top_indices[i % len(top_indices)]
        predict_2026.loc[idx, 'final_prediction'] += 1 if difference > 0 else -1

# 5. عرض التوقعات النهائية لأفضل 20 دولة
predictions_2026_sorted = predict_2026.sort_values(by='final_prediction', ascending=False)[['country_name', 'country_3_letter_code', 'prev_total_medals', 'final_prediction']].head(20)
predictions_2026_sorted.columns = ['الدولة', 'رمز الدولة', 'ميداليات 2022 (السابقة)', 'التوقع النهائي لـ 2026']
display(predictions_2026_sorted)

لم يتم العثور على ملف النموذج المحفوظ. جاري تدريب نموذج جديد وحفظه...
تم تدريب وحفظ النموذج الجديد بنجاح في: /content/winter_olympics_2026_model.pkl


,الدولة,رمز الدولة,ميداليات 2022 (السابقة),التوقع النهائي لـ 2026
515,Norway,NOR,37,27
259,Germany,GER,27,26
124,Canada,CAN,26,26
700,United States of America,USA,25,26
344,Italy,ITA,17,25
566,ROC,ROC,32,23
500,Netherlands,NED,17,16
362,Japan,JPN,18,13
233,France,FRA,14,13
144,People's Republic of China,CHN,15,13


In [4]:
import numpy as np
import pandas as pd
import joblib
import os
from google.colab import drive
import lightgbm as lgb

try:
    # 1. Mount Google Drive securely
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # Define the Drive directory paths for your dataset
    drive_hosts_path = '/content/drive/MyDrive/olympic_hosts.csv'
    drive_results_path = '/content/drive/MyDrive/olympic_results.csv'

    # Fallbacks in case files are located within an 'Olympic_Predictions' subfolder in Drive
    alt_drive_hosts = '/content/drive/MyDrive/Olympic_Predictions/olympic_hosts.csv'
    alt_drive_results = '/content/drive/MyDrive/Olympic_Predictions/olympic_results.csv'

    # Determine the correct files to load
    if os.path.exists(drive_hosts_path):
        hosts_file = drive_hosts_path
    elif os.path.exists(alt_drive_hosts):
        hosts_file = alt_drive_hosts
    else:
        hosts_file = '/content/olympic_hosts.csv'

    if os.path.exists(drive_results_path):
        results_file = drive_results_path
    elif os.path.exists(alt_drive_results):
        results_file = alt_drive_results
    else:
        results_file = '/content/olympic_results.csv'

    print(f"Loading hosts dataset from: {hosts_file}")
    print(f"Loading results dataset from: {results_file}")

    # Load datasets
    hosts_df = pd.read_csv(hosts_file)
    results_df = pd.read_csv(results_file)

    # 2. Process data
    winter_hosts = hosts_df[hosts_df['game_season'] == 'Winter'][['game_slug', 'game_year', 'game_location']]
    winter_results = results_df.merge(winter_hosts, left_on='slug_game', right_on='game_slug', how='inner')
    winter_results['medal_type'] = winter_results['medal_type'].fillna('No Medal')

    medal_counts = winter_results.groupby(['game_year', 'country_3_letter_code', 'country_name', 'medal_type']).size().unstack(fill_value=0).reset_index()
    for medal in ['GOLD', 'SILVER', 'BRONZE']:
        if medal not in medal_counts.columns:
            medal_counts[medal] = 0
    medal_counts['total_medals'] = medal_counts['GOLD'] + medal_counts['SILVER'] + medal_counts['BRONZE']

    host_countries_map = {1992: 'FRA', 1994: 'NOR', 1998: 'JPN', 2002: 'USA', 2006: 'ITA', 2010: 'CAN', 2014: 'RUS', 2018: 'KOR', 2022: 'CHN', 2026: 'ITA'}
    winter_data_sorted = medal_counts[medal_counts['game_year'] >= 1992].copy()
    winter_data_sorted['is_host'] = winter_data_sorted.apply(lambda row: 1 if host_countries_map.get(row['game_year']) == row['country_3_letter_code'] else 0, axis=1)
    winter_data_sorted = winter_data_sorted.sort_values(by=['country_3_letter_code', 'game_year']).reset_index(drop=True)
    winter_data_sorted['prev_total_medals'] = winter_data_sorted.groupby('country_3_letter_code')['total_medals'].shift(1).fillna(0)

    # Advanced feature engineering (rolling average)
    winter_data_sorted['rolling_avg_3_medals'] = winter_data_sorted.groupby('country_3_letter_code')['total_medals'].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
    ).fillna(0)

    # Macroeconomic indicators
    default_macro = {
        'USA': (23.3e12, 332e6), 'CHN': (17.7e12, 1412e6), 'GER': (4.26e12, 83.2e6), 'NOR': (482e9, 5.4e6),
        'CAN': (1.98e12, 38.2e6), 'RUS': (1.78e12, 143e6), 'ROC': (1.78e12, 143e6), 'FRA': (2.95e12, 67.7e6),
        'ITA': (2.1e12, 59.1e6), 'JPN': (4.93e12, 125.7e6), 'KOR': (1.81e12, 51.7e6), 'AUT': (480e9, 8.9e6),
        'SWE': (635e9, 10.4e6), 'SUI': (812e9, 8.7e6), 'NED': (1.01e12, 17.5e6), 'FIN': (297e9, 5.5e6)
    }

    winter_data_macro = winter_data_sorted.copy()
    def apply_macro(row):
        cc = row['country_3_letter_code']
        if cc in default_macro:
            gdp, pop = default_macro[cc]
        else:
            gdp, pop = 50e9, 10e6
        return pd.Series([gdp, pop, gdp/pop])

    winter_data_macro[['gdp', 'population', 'gdp_per_capita']] = winter_data_macro.apply(apply_macro, axis=1)
    winter_data_macro['log_gdp'] = np.log1p(winter_data_macro['gdp'])
    winter_data_macro['log_population'] = np.log1p(winter_data_macro['population'])
    winter_data_macro['log_gdp_per_capita'] = np.log1p(winter_data_macro['gdp_per_capita'])

    # 3. Load or Retrain Model
    local_ultimate_path = '/content/winter_olympics_2026_model_ultimate.pkl'
    drive_ultimate_path = '/content/drive/MyDrive/Olympic_Predictions/winter_olympics_2026_model_ultimate.pkl'
    ultimate_features = ['prev_total_medals', 'is_host', 'rolling_avg_3_medals', 'log_gdp', 'log_population', 'log_gdp_per_capita']

    if os.path.exists(local_ultimate_path):
        ultimate_model = joblib.load(local_ultimate_path)
        print("Successfully loaded local ultimate model.")
    elif os.path.exists(drive_ultimate_path):
        ultimate_model = joblib.load(drive_ultimate_path)
        print(f"Successfully loaded ultimate model from Drive: {drive_ultimate_path}")
    else:
        print("Re-training ultimate LightGBM model from data...")
        best_params = {
            'subsample': 0.6, 'num_leaves': 15, 'n_estimators': 200,
            'min_child_samples': 20, 'max_depth': 6, 'learning_rate': 0.05,
            'colsample_bytree': 0.6, 'random_state': 42, 'verbose': -1
        }
        ultimate_model = lgb.LGBMRegressor(**best_params)
        ultimate_model.fit(winter_data_macro[ultimate_features], winter_data_macro['total_medals'])
        joblib.dump(ultimate_model, local_ultimate_path)
        print("Model trained and saved locally.")

    # 4. Generate Predictions for 2026
    countries_2022 = winter_data_macro[winter_data_macro['game_year'] == 2022].copy()
    predict_2026 = pd.DataFrame({
        'country_3_letter_code': countries_2022['country_3_letter_code'],
        'country_name': countries_2022['country_name'],
        'prev_total_medals': countries_2022['total_medals'],
        'is_host': countries_2022['country_3_letter_code'].apply(lambda x: 1 if x == 'ITA' else 0),
        'rolling_avg_3_medals': countries_2022.apply(
            lambda row: winter_data_macro[
                (winter_data_macro['country_3_letter_code'] == row['country_3_letter_code']) &
                (winter_data_macro['game_year'] <= 2022)
            ]['total_medals'].tail(3).mean(),
            axis=1
        ).fillna(0),
        'log_gdp': countries_2022['log_gdp'],
        'log_population': countries_2022['log_population'],
        'log_gdp_per_capita': countries_2022['log_gdp_per_capita']
    })

    raw_preds_2026 = ultimate_model.predict(predict_2026[ultimate_features])
    predict_2026['raw_prediction'] = np.clip(raw_preds_2026, 0, None)

    target_sum = 116 * 3
    current_sum = predict_2026['raw_prediction'].sum()
    predict_2026['final_prediction'] = (predict_2026['raw_prediction'] / current_sum) * target_sum
    predict_2026['final_prediction'] = predict_2026['final_prediction'].round().astype(int)

    difference = target_sum - predict_2026['final_prediction'].sum()
    if difference != 0:
        top_indices = predict_2026.sort_values(by='final_prediction', ascending=False).index
        for i in range(abs(difference)):
            idx = top_indices[i % len(top_indices)]
            predict_2026.loc[idx, 'final_prediction'] += 1 if difference > 0 else -1

    # Display predictions
    leaderboard_2026 = predict_2026.sort_values(by='final_prediction', ascending=False).copy()
    leaderboard_display = leaderboard_2026.head(20)[[
        'country_name', 'country_3_letter_code', 'prev_total_medals', 'final_prediction'
    ]]
    leaderboard_display.columns = ['الدولة', 'رمز الدولة', 'ميداليات 2022', 'التوقع النهائي لـ 2026']

    print("\n=== جدول التوقعات النهائي لدورة الألعاب الأولمبية الشتوية 2026 ===")
    display(leaderboard_display)

except Exception as e:
    print(f"An error occurred: {e}")

Mounting Google Drive...
Mounted at /content/drive
Loading hosts dataset from: /content/olympic_hosts.csv
Loading results dataset from: /content/olympic_results.csv
Re-training ultimate LightGBM model from data...
Model trained and saved locally.

=== جدول التوقعات النهائي لدورة الألعاب الأولمبية الشتوية 2026 ===


,الدولة,رمز الدولة,ميداليات 2022,التوقع النهائي لـ 2026
700,United States of America,USA,25,32
515,Norway,NOR,37,32
259,Germany,GER,27,32
124,Canada,CAN,26,30
566,ROC,ROC,32,29
54,Austria,AUT,18,21
642,Sweden,SWE,18,20
625,Switzerland,SUI,14,19
500,Netherlands,NED,17,19
233,France,FRA,14,18


In [5]:
# عرض جدول التنبؤات النهائي المعتمد والمحسوب بالكامل
display(leaderboard_display)

,الدولة,رمز الدولة,ميداليات 2022,التوقع النهائي لـ 2026
700,United States of America,USA,25,32
515,Norway,NOR,37,32
259,Germany,GER,27,32
124,Canada,CAN,26,30
566,ROC,ROC,32,29
54,Austria,AUT,18,21
642,Sweden,SWE,18,20
625,Switzerland,SUI,14,19
500,Netherlands,NED,17,19
233,France,FRA,14,18
